In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]", "executable": "/home/miakhlov/course-production-verified-20261118/scripts/environment/.venv-base/bin/python", "platform": "Linux-6.18.58.1-microsoft-standard-WSL2-x86_64-with-glibc2.39", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0", "fqdn": "1.6.0", "h11": "0.16.0", "httpcore": "1.0.9", "httpx": "0.28.1", "idna": "3.20", "ipykernel": "7.4.0", "ipython": "9.18.1", "ipython_p

# Разбор A. Python как инструмент исследования

Паспорт входа есть, но таблица после обработки стала короче. Были ли потеряны неправильные строки или научно важные наблюдения? Исследуем таблицу ксантинов, затем перенесём проверку на формы B. Гипотеза: явная схема и журнал отклонений делают преобразование проверяемым, в отличие от молчаливого dropna. Базовые конструкции Python здесь служат научной прослеживаемости.

Создайте записи расчёта для 1.0 mg вещества в 10.0 mL; массы берём из структурных карточек. Это моделируемая подготовка, не протокол реального раствора. Напишите функцию с проверками конечности и положительности. Добавьте контроль на нулевой объём, сохраните valid и rejected отдельно и проверьте повторное чтение JSON.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M02-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
def concentration(mass_mg,molar_mass,volume_ml):
    values=[mass_mg,molar_mass,volume_ml]
    if not all(math.isfinite(v) and v>0 for v in values):raise ValueError('Positive finite inputs required')
    return (mass_mg/1000)/molar_mass/(volume_ml/1000)
rows=[{'name':n,'c_mol_L':concentration(1,compound(n)['molecular_weight_g_mol'],10)} for n in ['caffeine','istradefylline']]
display(rows)


[{'name': 'caffeine', 'c_mol_L': 0.0005149595756733096},
 {'name': 'istradefylline', 'c_mol_L': 0.00026014568158168577}]

Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
try:concentration(1,194.19,0)
except ValueError as error:print('Rejected control:',str(error))
else:raise AssertionError('Zero volume was accepted')
fig,ax=plt.subplots();ax.bar([r['name'] for r in rows],[r['c_mol_L'] for r in rows]);ax.set_ylabel('Simulated c / mol L^-1')
publish_plot(fig,'concentrations.png')


Rejected control: Positive finite inputs required


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
write_json(OUT/'table.json',rows)
assert json.loads((OUT/'table.json').read_text(encoding='utf-8'))==rows
result={'rows':rows,'kind':'simulation'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M02-L02','A',result,ROOT,inputs=['practicuum/shared/data/compounds.json'])
print('Versioned evidence:',receipt)


Versioned evidence: /home/miakhlov/course-production-verified-20261118/practicuum/evidence/20261118-full-course-linux/workspace/my_research/evidence/M02-L02/20261118T141932740830Z-A-0623bec223a340e0930ae832fb078e90.json


Количество знаков после запятой в выводе не создаёт точность весов. Для B масса безводной кислоты и соли на этикетке имеет разный смысл. Следующий модуль отвечает, какая структура стоит за строкой и формулой.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?